<a href="https://colab.research.google.com/github/YujiaLIAO-1/housing/blob/main/in_class_assignment7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [86]:
import pandas as pd

In [87]:
df = pd.read_csv("titanic.csv")

In [88]:
X = df[['Pclass','Sex','Age','Fare','Embarked']]
y = df['Survived']

In [89]:
X.isnull().sum()/len(df)*100

,0
Pclass,0.000000
Sex,0.000000
Age,19.865320
Fare,0.000000
Embarked,0.224467


In [90]:
df.dropna(subset=['Embarked'], inplace=True)

In [91]:
df.Age.fillna(df.Age.mean(), inplace=True)

<ipython-input-91-efce9d0f789a>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.Age.fillna(df.Age.mean(), inplace=True)


In [92]:
num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Embarked', 'Pclass']

In [93]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# 对数字进行标准化，对文字进行onehot encoding
num_pipeline = Pipeline(
    steps=[
        ('num_imputer', SimpleImputer()),
        ('scaler', StandardScaler()),
        ]
)

cat_pipeline = Pipeline(
    steps=[
        ('cat_imputer', SimpleImputer()),
        ('onehot', OneHotEncoder()),
    ]
)

In [94]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)


In [95]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, num_features),  # pipeline name, pipeline, features to process
        ('cat_pipeline', cat_pipeline, cat_features),  # pipeline name, pipeline, features to process
    ]
)

In [96]:
# 合并
from sklearn.tree import DecisionTreeClassifier

# final decision tree (dt) pipeline
dt_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('tree_clf', DecisionTreeClassifier()),
    ]
)

In [97]:
# GridSearch with 10-fold cross validation and accuracy as the metric
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        'preprocessor__num_pipeline__num_imputer__strategy': ['mean', 'median'],
        'preprocessor__cat_pipeline__cat_imputer__strategy': ['most_frequent'],  # only one choice for this parameter
        'tree_clf__criterion': ['gini', 'entropy', 'log_loss'],
        'tree_clf__splitter': ['best', 'random'],
        'tree_clf__max_depth': [3, 4, 5, 6, 7, 8, 9],
    }
]

# set up the grid search
grid_search = GridSearchCV(dt_pipeline, param_grid, cv=5, scoring='accuracy')

In [98]:
# train the model using the full pipeline
grid_search.fit(X_train, y_train)

# check the best performing parameter combination
grid_search.best_params_

{'preprocessor__cat_pipeline__cat_imputer__strategy': 'most_frequent',
 'preprocessor__num_pipeline__num_imputer__strategy': 'median',
 'tree_clf__criterion': 'entropy',
 'tree_clf__max_depth': 8,
 'tree_clf__splitter': 'random'}

In [99]:
grid_search.cv_results_['mean_test_score']

array([0.82185806, 0.82027097, 0.82664516, 0.82184516, 0.82187097,
       0.81059355, 0.81381935, 0.80903226, 0.82824516, 0.82180645,
       0.80898065, 0.81381935, 0.81061935, 0.83781935, 0.82185806,
       0.82023226, 0.82503226, 0.82668387, 0.82024516, 0.82345806,
       0.81220645, 0.81063226, 0.81863226, 0.79776774, 0.82344516,
       0.81539355, 0.83310968, 0.79783226, 0.82185806, 0.81381935,
       0.82503226, 0.81540645, 0.82023226, 0.81700645, 0.81381935,
       0.82505806, 0.82025806, 0.82025806, 0.82664516, 0.82985806,
       0.82987097, 0.81225806, 0.82185806, 0.82350968, 0.82985806,
       0.82343226, 0.82989677, 0.82180645, 0.81544516, 0.82343226,
       0.83468387, 0.82825806, 0.81380645, 0.81865806, 0.81863226,
       0.81063226, 0.82185806, 0.82667097, 0.82503226, 0.80580645,
       0.82184516, 0.81868387, 0.81060645, 0.82501935, 0.82181935,
       0.83467097, 0.82021935, 0.84109677, 0.82665806, 0.81541935,
       0.82185806, 0.80745806, 0.82341935, 0.81859355, 0.81861

In [100]:
tree_clf_best = grid_search.best_estimator_

In [101]:
# final evaluation using the test data
y_pred = tree_clf_best.predict(X_test)

# calculate accuracy, precision, recall, f1-score
# y_test is the ground truth, y_pred is our model's prediction
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

print(f'Accuracy Score : {accuracy_score(y_test, y_pred)}')
print(f'Precision Score : {precision_score(y_test, y_pred)}')
print(f'Recall Score : {recall_score(y_test, y_pred)}')
print(f'F1 Score : {f1_score(y_test, y_pred)}')


Accuracy Score : 0.7798507462686567
Precision Score : 0.7435897435897436
Recall Score : 0.5979381443298969
F1 Score : 0.6628571428571428


In [102]:
titanic_test = pd.read_csv('test.csv')
titanic_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.0+ KB


In [103]:
y_pred_titanic = tree_clf_best.predict(titanic_test)
# combine id and prediction for kaggle submission
dt_pipeline_submit = pd.DataFrame({
    'PassengerId': titanic_test['PassengerId'],
    'Survived': y_pred_titanic
})

# generate the csv
dt_pipeline_submit.to_csv('inclass7-submit.csv', index=False)